<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; EDM&plus; Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">EDM&plus; NB06 &mdash; Luminesce Reporting</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Runs SQL reports over everything built in NB01 to NB05 using LUSID's Luminesce data virtualisation engine.</div>
</div>

<sub>EDM&plus; pack sequence: NB01 &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; <b>NB06</b> &nbsp;&rarr;&nbsp; NB07</sub>

# NB06: Luminesce Reporting

**What this does:** Runs SQL queries against everything built in NB01 through NB05.

**Business context:** Luminesce is LUSID's SQL data virtualisation engine. Operations teams use it for reporting, data extracts, and reconciliation. You can also paste these queries into the Data Virtualisation page in the LUSID web app.

**Key learning:** The `Lusid.Instrument.Property` provider needs a specific InstrumentId. For cross-instrument aggregation, use `Lusid.Portfolio.Holding` instead.

**Verify after running:** Go to Data Virtualisation in the sidebar and paste any query.

In [ ]:
# Run this cell first — installs required packages (takes ~30 seconds, only needed once per session)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'lusid-sdk', 'lusid-workflow-sdk', 'finbourne-sdk-utilities', 'lusid-drive-sdk', 'lumipy'])
print("✅ Packages installed")

In [ ]:
# ============================================================================
# CREDENTIALS: edit secrets.json (NOT this notebook)
# ============================================================================
# Copy secrets.json.example to secrets.json and fill in your domain + PAT:
#   {
#     "api_url": "https://<YOUR_DOMAIN>.lusid.com/api",
#     "personal_access_token": "<YOUR_PAT>"
#   }
# To get a PAT: LUSID web app → profile icon (top right) → Personal Access Tokens → Create
# (Override the file location with the EDM_SECRETS_PATH environment variable.)

import json as _json, os as _os
_secrets_path = _os.environ.get("EDM_SECRETS_PATH", "secrets.json")
with open(_secrets_path) as _f:
    _secrets = _json.load(_f)
api_url = _secrets["api_url"]
personal_access_token = _secrets["personal_access_token"]

# ============================================================================
# Imports — do not edit below this line
# ============================================================================
import os, json
from datetime import datetime, timedelta, date, timezone
import datetime as dt
import pandas as pd
import pytz

import lusid as lu
import lusid.models as lm

try:
    import lusid_workflow as lwf
    import lusid_workflow.models as wf_models
except ImportError:
    lwf = None
    wf_models = None
    print("⚠️ lusid_workflow not available")

import lumipy as lp

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.4f}".format

# ============================================================================
# Authentication
# ============================================================================
if "<YOUR_DOMAIN>" in api_url or "<YOUR_PAT>" in personal_access_token or not personal_access_token:
    raise ValueError(
        "\n\n⛔ You need to edit the two lines at the top of this cell:\n"
        "   1. Replace <YOUR_DOMAIN> with your LUSID domain name\n"
        "   2. Replace <YOUR_PAT> with your Personal Access Token\n\n"
        "   To get a PAT: LUSID web app → profile icon → Personal Access Tokens → Create")

def get_factory(app='lusid'):
    module = __import__(app)
    # Each service has a different URL path
    url_map = {'lusid': '/api', 'lusid_workflow': '/workflow', 'lusid_drive': '/drive'}
    service_url = api_url.replace('/api', url_map.get(app, '/api'))
    config_loaders = [module.extensions.ArgsConfigurationLoader(
        api_url=service_url, access_token=personal_access_token)]
    return module.extensions.SyncApiClientFactory(config_loaders=config_loaders)

def get_lumi():
    luminesce_url = api_url.replace('/api', '/honeycomb')
    return lp.get_client(access_token=personal_access_token, api_url=luminesce_url)

# Initialise
lusid_factory = get_factory('lusid')

# Verify connection
try:
    api_instance = lusid_factory.build(lu.ApplicationMetadataApi)
    api_response = api_instance.get_lusid_versions()
    domain = api_response.links[0].href
    env_url = domain[0:domain.find('/app/')]
    print(f"✅ Connected to {env_url}")
    print(f"   API Version: {api_response.build_version}")
except Exception as e:
    print(f"⚠️ Connected but could not verify: {e}")

lumi = get_lumi()
print("✅ Luminesce ready")

---
## Report 1: Instrument Universe

In [ ]:
print("--- Report 1: Instrument Universe ---")
query = '''
SELECT i.DisplayName, i.ClientInternal, i.Isin, i.Figi, i.Sedol
FROM Lusid.Instrument i
WHERE i.ClientInternal LIKE 'EQT%' OR i.ClientInternal LIKE 'IVG%'
   OR i.ClientInternal LIKE 'GOV%' OR i.ClientInternal LIKE 'ETF%'
ORDER BY i.DisplayName LIMIT 20
'''
try:
    df = lumi.run(query, quiet=True)
    display(df)
except Exception as e:
    print(f"Error: {e}")

---
## Report 2: Holdings by Portfolio

In [ ]:
print("--- Report 2: Holdings Summary ---")
query = '''
SELECT h.PortfolioCode, COUNT(*) as HoldingCount, SUM(h.Units) as TotalUnits
FROM Lusid.Portfolio.Holding h
WHERE h.PortfolioScope = 'edm-training2-accounts' AND h.EffectiveAt = #2026-06-01#
GROUP BY h.PortfolioCode ORDER BY h.PortfolioCode
'''
try:
    df = lumi.run(query, quiet=True)
    display(df)
except Exception as e:
    print(f"Error: {e}")

---
## Report 3: Instrument Properties
Note: `Lusid.Instrument.Property` requires a specific InstrumentId. Change the code below to check different instruments.

In [ ]:
print("--- Report 3: Apple (EQT00001) Properties ---")
query = '''
SELECT p.PropertyCode, p.Value
FROM Lusid.Instrument.Property p
WHERE p.InstrumentIdType = 'ClientInternal' AND p.InstrumentId = 'EQT00001'
  AND p.PropertyScope = 'edm-training2'
'''
try:
    df = lumi.run(query, quiet=True)
    display(df)
except Exception as e:
    print(f"Error: {e}")

---
## Report 4: Holdings with Instrument Enrichment
Joins holdings with the instrument master to bring in identifiers.

In [ ]:
print("--- Report 4: Global Equities Holdings Enriched ---")
query = '''
SELECT h.PortfolioCode, h.DisplayName, h.Units, h.CostAmount, h.CostCurrency,
       i.ClientInternal, i.Isin, i.Figi
FROM Lusid.Portfolio.Holding h
LEFT JOIN Lusid.Instrument i ON h.LusidInstrumentId = i.LusidInstrumentId
WHERE h.PortfolioScope = 'edm-training2-accounts' AND h.EffectiveAt = #2026-06-01#
  AND h.PortfolioCode = 'TPV_GlobalEquities'
ORDER BY h.Units DESC
'''
try:
    df = lumi.run(query, quiet=True)
    display(df)
except Exception as e:
    print(f"Error: {e}")

print("\n✅ NB06 Complete")